In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/raw/cambio_ipca_ic-br_mensal_2008_2023.csv",
    sep="\t"
)

df.columns = [
    "data",
    "cambio",
    "ipca",
    "icbr"
]

# converter números brasileiros
for col in ["cambio","ipca","icbr"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",",".")
        .astype(float)
    )

# data
df["data"] = pd.to_datetime(df["data"], format="%m/%Y")

df["ano"] = df["data"].dt.year
df["mes"] = df["data"].dt.month


In [13]:
selic = pd.read_csv(
    "../data/raw/selic_diaria_2008_2023.csv",
    sep="\t"
)

selic.describe()

selic.columns = ["data","selic"]

selic["selic"] = (
    selic["selic"]
    .astype(str)
    .str.replace(",",".")
    .astype(float)
)

selic["data"] = pd.to_datetime(selic["data"], format="%d/%m/%Y")

selic["ano"] = selic["data"].dt.year

selic_anual = (
    selic
    .groupby("ano")["selic"]
    .mean()
    .reset_index()
)

In [14]:
macro_mensal = df.copy()

macro_anual = (
    macro_mensal
    .groupby("ano")
    .agg({
        "cambio":"mean",
        "icbr":"mean"
    })
)

# ipca = dezembro
ipca_dez = (
    macro_mensal[macro_mensal["mes"] == 12]
    .set_index("ano")["ipca"]
)

macro_anual["ipca"] = ipca_dez

macro_anual = macro_anual.reset_index()

incc

In [23]:
# ===============================
# INCC (custo da construção)
# ===============================

incc = pd.read_csv("../data/processed/incc-mensal.csv")

incc.columns = ["data", "incc"]

# limpar aspas e converter
incc["incc"] = (
    incc["incc"]
    .astype(str)
    .str.replace("'", "")
    .astype(float)
)

# converter data
incc["data"] = pd.to_datetime(incc["data"], format="%m/%Y")

incc["ano"] = incc["data"].dt.year
incc["mes"] = incc["data"].dt.month

# ===============================
# INCC anual (dezembro = 12 meses)
# ===============================

incc_dez = (
    incc[incc["mes"] == 12][["ano", "incc"]]
)

# ===============================
# MERGE SEGURO (evita erro de index)
# ===============================

macro_anual = macro_anual.merge(
    incc_dez,
    on="ano",
    how="left"
)

# ===============================
# TRANSFORMAÇÕES
# ===============================

macro_anual["log_incc"] = np.log(macro_anual["incc"])
macro_anual["dlog_incc"] = macro_anual["log_incc"].diff()

# (opcional, mas recomendado)
macro_anual["dlog_incc_lag1"] = macro_anual["dlog_incc"].shift(1)

# ===============================
# FINAL
# ===============================

macro = macro_anual.copy()

df_final = df.merge(
    macro,
    on="ano",
    how="left"
)

In [27]:
macro = macro_anual.merge(
    selic_anual,
    on="ano",
    how="left"
)

macro = macro.sort_values("ano")

macro["icbr_var"] = macro["icbr"].pct_change()

macro = macro[
    (macro["ano"] >= 2009) &
    (macro["ano"] <= 2023)
]


macro = macro.rename(columns={
    "cambio":"cambio_brl_usd",
    "icbr":"ic_br",
    "ipca":"ipca_12m",
    "selic":"selic_media",
    "icbr_var":"d_ic_br"
})

macro.to_csv(
    "../data/processed/macro_brasil_anual_2009_2023.csv",
    index=False
)


In [28]:
macro.describe()

,ano,cambio_brl_usd,ic_br,ipca_12m,incc_x,d_incc,log_incc,dlog_incc,incc_y,incc,dlog_incc_lag1,selic_media,d_ic_br
count,15.000000,15.000000,15.000000,15.000000,0.0,0.0,14.000000,13.000000,14.000000,14.000000,13.000000,15.000000,15.000000
mean,2016.000000,3.348151,193.816500,5.856000,NaN,NaN,6.483817,0.070411,678.016000,678.016000,0.070411,9.558275,0.097491
std,4.472136,1.348503,101.078483,2.116965,NaN,NaN,0.276081,0.024227,188.178431,188.178431,0.024227,3.387496,0.157491
min,2009.000000,1.674967,89.556667,2.950000,NaN,NaN,6.042754,0.037679,421.051000,421.051000,0.037679,2.886612,-0.126418
25%,2012.500000,2.077608,122.696250,4.415000,NaN,NaN,6.278034,0.059454,533.046750,533.046750,0.059454,7.436644,0.018042
50%,2016.000000,3.331533,162.441667,5.840000,NaN,NaN,6.505492,0.072148,669.100500,669.100500,0.072148,10.143151,0.077651
75%,2019.500000,4.470025,211.450000,6.350000,NaN,NaN,6.645058,0.077834,769.093250,769.093250,0.077834,12.151712,0.185043
max,2023.000000,5.395025,411.879167,10.670000,NaN,NaN,6.958099,0.129694,1051.632000,1051.632000,0.129694,14.178962,0.499607
